# Doo — SVC 모델

고객 이탈 예측을 위한 RBF 커널 SVC 분류 모델입니다.

- Train 데이터로만 모델을 학습합니다.
- Validation 데이터로 성능과 임계값을 비교합니다.
- Test 데이터는 최종 모델 확정 전까지 사용하지 않습니다.
- 팀 공통 시드 `random_state=42`를 사용합니다.

In [1]:
from pathlib import Path
import pickle

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "models":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "preprocessed"
X_train = pd.read_csv(DATA_DIR / "X_train.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv")["churn"]
X_val = pd.read_csv(DATA_DIR / "X_val.csv")
y_val = pd.read_csv(DATA_DIR / "y_val.csv")["churn"]

with open(DATA_DIR / "preprocessor.pkl", "rb") as file:
    preprocessor = pickle.load(file)

print("X_train:", X_train.shape, "| X_val:", X_val.shape)
print("이탈률 - train:", y_train.mean().round(3), "| val:", y_val.mean().round(3))

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\munkyu\\Desktop\\skn33-2nd-5team\\models\\jhd\\data\\preprocessed\\X_train.csv'

## 1. 데이터 확인

SVC는 피처 간 거리와 마진을 사용하므로 스케일링이 중요합니다. 현재 데이터는 팀 전처리 과정에서 이미 스케일링되었습니다.

In [ ]:
display(X_train.head())
display(pd.DataFrame({
    "dtype": X_train.dtypes,
    "train_missing": X_train.isna().sum(),
    "val_missing": X_val.isna().sum(),
}))
display(y_train.value_counts().rename_axis("churn").to_frame("count"))

## 2. 모델 학습

RBF 커널을 사용해 선형 모델이 표현하기 어려운 비선형 결정경계를 학습합니다. 확률이 필요한 평가는 `CalibratedClassifierCV`로 SVC의 점수를 보정해 사용합니다.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import SVC

svc_base_model = SVC(
    C=1.0,
    kernel="rbf",
    gamma="scale",
    class_weight="balanced",
    random_state=42,
)

# scikit-learn 1.9부터 SVC(probability=True)가 deprecated되어
# 권장 방식인 확률 보정 모델을 사용한다.
svc_model = CalibratedClassifierCV(
    estimator=svc_base_model,
    method="sigmoid",
    cv=5,
    ensemble=False,
)

svc_model.fit(X_train, y_train)
print("학습 완료")

## 3. Validation 기본 성능

기본 분류 임계값 0.5에서 Train과 Validation 성능 차이 및 주요 분류 지표를 확인합니다.

In [ ]:
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)

svc_train_pred = svc_model.predict(X_train)
svc_val_pred = svc_model.predict(X_val)
svc_val_proba = svc_model.predict_proba(X_val)[:, 1]

svc_metrics_05 = pd.Series({
    "train_accuracy": accuracy_score(y_train, svc_train_pred),
    "val_accuracy": accuracy_score(y_val, svc_val_pred),
    "recall": recall_score(y_val, svc_val_pred),
    "precision": precision_score(y_val, svc_val_pred),
    "f1": f1_score(y_val, svc_val_pred),
    "roc_auc": roc_auc_score(y_val, svc_val_proba),
}, name="threshold_0.50")

display(svc_metrics_05.to_frame())
print("Confusion matrix")
print(confusion_matrix(y_val, svc_val_pred))
print("\nClassification report")
print(classification_report(y_val, svc_val_pred, digits=3))

## 4. 임계값 비교

Validation 데이터에서 Recall, Precision, F1-score와 이탈 예측 고객 수를 비교합니다.

In [ ]:
svc_threshold_results = []

for threshold in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
    svc_threshold_pred = (svc_val_proba >= threshold).astype(int)
    svc_threshold_results.append({
        "threshold": threshold,
        "recall": recall_score(y_val, svc_threshold_pred),
        "precision": precision_score(y_val, svc_threshold_pred, zero_division=0),
        "f1": f1_score(y_val, svc_threshold_pred, zero_division=0),
        "predicted_churn_count": int(svc_threshold_pred.sum()),
    })

svc_threshold_df = pd.DataFrame(svc_threshold_results)
display(svc_threshold_df.style.format({
    "threshold": "{:.2f}", "recall": "{:.3f}",
    "precision": "{:.3f}", "f1": "{:.3f}",
}))

## 5. Support Vector와 결정함수 확인

Support Vector는 결정경계를 만드는 데 직접 영향을 주는 학습 샘플입니다. 결정함수 값이 0에 가까울수록 경계에 가까운 고객입니다.

In [ ]:
# ensemble=False에서는 전체 Train 데이터로 다시 학습된 SVC가 한 개 저장된다.
svc_fitted_base_model = svc_model.calibrated_classifiers_[0].estimator

svc_support_summary = pd.Series({
    "total_train_samples": len(X_train),
    "support_vectors": len(svc_fitted_base_model.support_),
    "support_vector_ratio": len(svc_fitted_base_model.support_) / len(X_train),
    "stay_support_vectors": svc_fitted_base_model.n_support_[0],
    "churn_support_vectors": svc_fitted_base_model.n_support_[1],
})
svc_val_decision_score = svc_fitted_base_model.decision_function(X_val)
display(svc_support_summary.to_frame("value"))

## 6. 혼동행렬 시각화

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

svc_confusion_matrix = confusion_matrix(y_val, svc_val_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(
    svc_confusion_matrix, annot=True, fmt="d", cmap="Purples",
    xticklabels=["Stay (0)", "Churn (1)"],
    yticklabels=["Stay (0)", "Churn (1)"],
)
plt.xlabel("Predicted label")
plt.ylabel("Actual label")
plt.title("SVC Confusion Matrix (Threshold = 0.50)")
plt.tight_layout()
plt.show()

## 7. 임계값별 성능 변화

In [ ]:
plt.figure(figsize=(9, 5))
for metric, color in [("recall", "tab:red"), ("precision", "tab:blue"), ("f1", "tab:green")]:
    plt.plot(
        svc_threshold_df["threshold"], svc_threshold_df[metric],
        marker="o", linewidth=2, color=color,
        label=metric.upper() if metric == "f1" else metric.title(),
    )
plt.axhline(0.80, color="gray", linestyle=":", label="Recall target: 0.80")
plt.xlabel("Classification threshold")
plt.ylabel("Score")
plt.title("SVC Metrics by Threshold")
plt.ylim(0, 1)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

## 8. ROC Curve와 Precision–Recall Curve

In [ ]:
from sklearn.metrics import auc, average_precision_score, precision_recall_curve, roc_curve

svc_fpr, svc_tpr, _ = roc_curve(y_val, svc_val_proba)
svc_roc_auc = auc(svc_fpr, svc_tpr)
svc_pr_precision, svc_pr_recall, _ = precision_recall_curve(y_val, svc_val_proba)
svc_average_precision = average_precision_score(y_val, svc_val_proba)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(svc_fpr, svc_tpr, linewidth=2, label=f"ROC-AUC = {svc_roc_auc:.3f}")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random classifier")
axes[0].set(xlabel="False Positive Rate", ylabel="True Positive Rate", title="ROC Curve")
axes[0].legend()
axes[0].grid(alpha=0.25)
axes[1].plot(
    svc_pr_recall, svc_pr_precision, linewidth=2, color="tab:orange",
    label=f"Average Precision = {svc_average_precision:.3f}",
)
axes[1].axhline(y_val.mean(), linestyle="--", color="gray", label=f"Baseline = {y_val.mean():.3f}")
axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision–Recall Curve")
axes[1].legend()
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 9. 결정함수 점수 분포

두 클래스의 결정함수 분포가 멀리 떨어질수록 구분이 잘 되는 모델입니다. 0 주변에 겹치는 고객은 분류가 어려운 사례입니다.

In [ ]:
svc_decision_df = pd.DataFrame({
    "decision_score": svc_val_decision_score,
    "actual_churn": y_val.to_numpy(),
})
plt.figure(figsize=(9, 5))
sns.histplot(
    data=svc_decision_df, x="decision_score", hue="actual_churn",
    bins=35, kde=True, element="step", stat="density", common_norm=False,
)
plt.axvline(0, color="black", linestyle="--", label="Decision boundary")
plt.title("SVC Decision Score Distribution")
plt.xlabel("Decision score")
plt.tight_layout()
plt.show()

## 10. 실험 결과 기록

Recall 0.80 이상 후보 중 F1-score가 가장 높은 임계값을 확인합니다.

In [ ]:
svc_candidates = svc_threshold_df[svc_threshold_df["recall"] >= 0.80]
if svc_candidates.empty:
    print("Recall 0.80 이상인 임계값 후보가 없습니다.")
else:
    svc_best_candidate = svc_candidates.loc[svc_candidates["f1"].idxmax()]
    display(svc_best_candidate.to_frame("selected_value"))

## 11. 모델 구성 및 평가 기준 설명

이 노트북은 수업에서 배운 SVC의 **스케일링, 커널, C, gamma, Support Vector와 결정함수** 개념을 고객 이탈 예측 문제에 적용했다.

### 데이터와 모델 선택

- SVC는 거리 기반 모델이므로 스케일링이 중요하다. 팀장이 Train 데이터로 학습한 공통 전처리 결과를 사용하고 추가 스케일링은 하지 않았다.
- `kernel='rbf'`는 선형 모델로 표현하기 어려운 비선형 관계를 학습한다.
- `C=1.0`은 오분류 벌점의 기본 기준이고, `gamma='scale'`은 피처 수와 분산을 기준으로 영향 범위를 자동 설정한다.
- scikit-learn 1.9부터 `SVC(probability=True)`가 deprecated되어 `CalibratedClassifierCV`로 확률을 보정한다. `method='sigmoid'`, `cv=5`, `ensemble=False`를 사용해 교차검증 기반 확률을 만들고 전체 Train 데이터로 최종 SVC를 다시 학습한다.
- `class_weight='balanced'`는 클래스 비율을 반영한다. 현재 데이터는 거의 균형이므로 추후 `None`과 비교할 수 있다.

### 평가 기준

실제 이탈 고객을 놓치지 않기 위해 Recall을 우선하되 Precision, F1-score, ROC-AUC를 함께 확인한다. 기본 임계값 외에 0.30~0.70을 비교하고 Test 데이터는 최종 모델 확정 전까지 사용하지 않는다. SVC는 트리 기반 피처 중요도를 제공하지 않으므로 Support Vector 수와 결정함수 분포를 확인한다. 최종 선정은 Logistic Regression, Random Forest, XGBoost와 동일한 Validation 기준으로 비교해 결정한다.